## 🎯 Learning Objectives
* Understand the core components and flow of the Reinforcement Learning from Human Feedback (RLHF) pipeline.
* Grasp the purpose and methodology of Reward Modeling (RM) in the context of LLM alignment.
* Comprehend how Proximal Policy Optimization (PPO) is applied to finetune LLMs using the learned reward model.
* Identify the practical considerations and challenges in implementing RLHF for large language models.


## FT03-L08: RLHF Pipeline: Reward Modelling and PPO Finetuning

Welcome to a pivotal lesson in mastering Reinforcement Learning for advanced AI systems. Today, we delve into the **Reinforcement Learning from Human Feedback (RLHF) pipeline**, the cornerstone technique for aligning modern Large Language Models (LLMs) with human values, preferences, and instructions. This process is what transforms powerful, but often unaligned, base models into helpful, harmless, and honest AI assistants.

Imagine you're a master chef, and you've just created a new dish. While you know the ingredients and cooking techniques, you don't truly know if it's a *great* dish until your customers taste it and provide feedback. RLHF is precisely this feedback loop for LLMs. Instead of a chef, we have a pre-trained LLM; instead of customers, we have human annotators; and instead of a dish, we have the LLM's generated text.

The RLHF pipeline typically involves three main steps, building upon an initial pre-trained LLM:

1.  **Supervised Finetuning (SFT)**: (Often a precursor, not strictly part of RLHF but sets the stage). A pre-trained LLM is finetuned on a dataset of high-quality human-written demonstrations (prompt-response pairs). This helps the model learn to follow instructions and generate coherent, helpful responses in a supervised manner. Think of this as the chef learning basic recipes from a cookbook.

2.  **Reward Modelling (RM)**: This is where human feedback truly enters the picture. Instead of directly telling the LLM what to do, we show it multiple possible responses to a given prompt and ask humans to rank them by preference (e.g., which response is better, more helpful, less toxic). This preference data is then used to train a separate **Reward Model**. This RM is essentially a critic that learns to predict human preferences, assigning a scalar 'reward' to any given LLM response. It learns to quantify 'goodness' from human examples. This is like the chef's assistant learning to predict customer satisfaction based on past feedback.

    *   **Data Collection**: Humans compare and rank multiple model outputs for a given prompt.
    *   **Model Training**: A neural network (often a smaller version of the base LLM or a specialized architecture) is trained to output a scalar score for a given prompt-response pair, reflecting human preference. The training objective is typically to maximize the score difference between preferred and dispreferred responses.

3.  **PPO Finetuning (Policy Optimization)**: With a trained Reward Model in hand, we can now use Reinforcement Learning to further finetune the original LLM (now called the 'policy' model). The policy model generates responses, and the Reward Model evaluates these responses, providing a reward signal. The goal is to update the policy model's parameters to maximize the reward it receives from the RM. **Proximal Policy Optimization (PPO)** is a popular and robust on-policy algorithm used for this step.

    *   **Interaction**: The policy LLM generates responses to prompts.
    *   **Reward Calculation**: The Reward Model evaluates these responses and assigns a scalar reward.
    *   **Policy Update**: PPO uses this reward signal to update the policy LLM's weights, encouraging it to generate responses that the RM (and thus, implicitly, humans) would prefer. A key aspect of PPO is its ability to make stable updates by preventing the policy from changing too drastically in a single step, ensuring training stability.

This iterative process of generating responses, getting human feedback (via the RM), and updating the policy allows LLMs to learn complex, nuanced human preferences that are difficult to encode through simple supervised learning. By 2026, RLHF, often augmented with techniques like Direct Preference Optimization (DPO) or other preference-based learning methods, remains the gold standard for aligning powerful foundation models.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from trl import PPOTrainer, PPOConfig
from datasets import Dataset
import random

# --- Configuration for 2026-ready models and libraries ---
# In 2026, we'd likely use more advanced models, but for demonstration,
# we'll use readily available smaller models to keep it runnable.
# For production, consider models like Llama-3-8B-Instruct, Mistral-7B-Instruct-v0.3, etc.

# 1. Reward Model Training (Simplified Demonstration)
print("\n--- Step 1: Reward Model Training (Simplified) ---")

# A. Simulate Human Preference Data
# In a real scenario, this data would come from human annotators ranking LLM outputs.
# Each entry: (prompt, chosen_response, rejected_response)
preference_data = [
    {"prompt": "Write a short story about a brave knight.",
     "chosen": "Sir Reginald, with his shining armor, bravely faced the dragon, saving the village.",
     "rejected": "A knight went to fight a dragon. He won. The end."},
    {"prompt": "Explain quantum entanglement simply.",
     "chosen": "Imagine two coins, if one is heads, the other is tails, no matter how far apart. That's entanglement.",
     "rejected": "Quantum entanglement is a quantum mechanical phenomenon where two or more particles are linked."},
    {"prompt": "What is the capital of France?",
     "chosen": "The capital of France is Paris.",
     "rejected": "France's capital city is London."}
]

# B. Load a base model for the Reward Model (e.g., a small BERT-like model)
# In practice, the RM is often a finetuned version of the base LLM or a similar architecture.
rm_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# Add a classification head for binary preference (chosen vs. rejected)
rm_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=1)

# C. Prepare data for RM training
# The RM learns to assign a higher score to 'chosen' responses than 'rejected' ones.
# We'll create pairs (text, label) where label is 1 for chosen, 0 for rejected.
rm_training_data = []
for item in preference_data:
    # Concatenate prompt and response for the RM input
    rm_training_data.append({"text": item["prompt"] + " " + item["chosen"], "label": 1.0})
    rm_training_data.append({"text": item["prompt"] + " " + item["rejected"], "label": 0.0})

# Convert to a Hugging Face Dataset (simplified for demonstration)
rm_dataset = Dataset.from_list(rm_training_data)

# Tokenize the dataset
def tokenize_function(examples):
    return rm_tokenizer(examples["text"], truncation=True, max_length=128)

rm_tokenized_dataset = rm_dataset.map(tokenize_function, batched=True)
rm_tokenized_dataset = rm_tokenized_dataset.remove_columns(["text"])
rm_tokenized_dataset.set_format("torch")

# D. Simulate a single training step for the Reward Model
# In a real scenario, this would be a full training loop with an optimizer, epochs, etc.
print("Simulating a single Reward Model training step...")
optimizer = torch.optim.AdamW(rm_model.parameters(), lr=1e-5)
loss_fn = torch.nn.BCEWithLogitsLoss()

# Take a batch from the dataset
if len(rm_tokenized_dataset) > 0:
    batch = rm_tokenized_dataset.select(range(min(2, len(rm_tokenized_dataset)))) # Take first 2 samples
    inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]}
    labels = batch["label"].unsqueeze(-1) # Ensure labels are float and correct shape

    optimizer.zero_grad()
    outputs = rm_model(**inputs)
    logits = outputs.logits
    loss = loss_fn(logits, labels)
    loss.backward()
    optimizer.step()
    print(f"  Reward Model simulated loss: {loss.item():.4f}")
    print(f"  Reward Model output for chosen (example): {rm_model(**rm_tokenizer(preference_data[0]['prompt'] + ' ' + preference_data[0]['chosen'], return_tensors='pt')).logits.item():.2f}")
    print(f"  Reward Model output for rejected (example): {rm_model(**rm_tokenizer(preference_data[0]['prompt'] + ' ' + preference_data[0]['rejected'], return_tensors='pt')).logits.item():.2f}")
else:
    print("  No data for RM training simulation.")

# Mock the reward model's prediction function for PPO
# In a real scenario, this would be `rm_model.forward()`
def get_reward(prompt, response):
    # A very simplified mock: higher reward for 'brave' or 'Paris', lower for 'London'
    if "brave" in response.lower() or "paris" in response.lower():
        return torch.tensor(random.uniform(0.8, 1.2))
    elif "london" in response.lower() or "rejected" in response.lower():
        return torch.tensor(random.uniform(-0.5, 0.2))
    else:
        return torch.tensor(random.uniform(0.3, 0.7))


# 2. PPO Finetuning (Conceptual Demonstration)
print("\n--- Step 2: PPO Finetuning (Conceptual) ---")

# A. Load a pre-trained LLM (the 'policy' model)
# For a real RLHF setup, this would be an SFT-tuned model.
policy_model_name = "gpt2"
policy_tokenizer = AutoTokenizer.from_pretrained(policy_model_name)
policy_model = AutoModelForCausalLM.from_pretrained(policy_model_name)

# GPT2 tokenizer doesn't have a pad token by default, which is needed for batching.
if policy_tokenizer.pad_token is None:
    policy_tokenizer.pad_token = policy_tokenizer.eos_token

# B. Define PPO Configuration
ppo_config = PPOConfig(
    learning_rate=1e-5,
    batch_size=1,
    mini_batch_size=1,
    gradient_accumulation_steps=1,
    ppo_epochs=1,
    target_kl=0.01,
    seed=42,
    log_with=None # Disable logging for this simple demo
)

# C. Create a PPOTrainer
# In a real scenario, `ref_model` would be a frozen copy of the initial policy model
# to calculate KL divergence for stability.
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=None, # For simplicity, we omit ref_model in this conceptual demo
    tokenizer=policy_tokenizer,
    num_shared_layers=None # Not applicable for this simple setup
)

# D. Simulate a single PPO optimization step
print("Simulating a single PPO optimization step...")

# 1. Generate a prompt
input_prompt = "Write a short, positive affirmation:"
input_ids = policy_tokenizer(input_prompt, return_tensors="pt").input_ids

# 2. Policy generates a response
# In a real PPO loop, multiple responses would be generated.
# We'll use a simple generation for demonstration.
policy_model.eval() # Set to eval mode for generation
with torch.no_grad():
    generated_output = policy_model.generate(
        input_ids,
        max_new_tokens=20,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        pad_token_id=policy_tokenizer.eos_token_id
    )

response_ids = generated_output[:, input_ids.shape[-1]:]
response_text = policy_tokenizer.decode(response_ids[0], skip_special_tokens=True)
full_text = policy_tokenizer.decode(generated_output[0], skip_special_tokens=True)

print(f"  Prompt: {input_prompt}")
print(f"  Generated Response: {response_text}")

# 3. Get reward from the (mock) Reward Model
reward = get_reward(input_prompt, response_text)
print(f"  Mock Reward for response: {reward.item():.2f}")

# 4. Prepare for PPO update (conceptual)
# In a real PPO loop, we'd collect a batch of (query, response, reward) tuples.
# Here, we'll just show the structure for a single example.

# The PPOTrainer expects a list of dictionaries for its `step` method.
# Each dict should contain 'query', 'response', and 'rewards'.
# 'query' is the input_ids of the prompt.
# 'response' is the input_ids of the generated response.
# 'rewards' is a tensor of rewards.

# For this conceptual demo, we'll create a dummy batch for `ppo_trainer.step`
# In a real scenario, this would be a batch of multiple queries and responses.

# We need to ensure the `ppo_trainer` has a `ref_model` or handle its absence.
# For a truly minimal runnable example, we'll bypass `ppo_trainer.step`
# and just show the conceptual inputs it would take.

print("  PPO Trainer would now take (query_tensors, response_tensors, rewards) and perform an update.")
print("  Conceptual PPO input structure:")
print(f"    Query Tensor (input_ids): {input_ids.shape}")
print(f"    Response Tensor (generated_ids): {response_ids.shape}")
print(f"    Reward Tensor: {reward.shape}")

# A full `ppo_trainer.step` call would look like this (if `ref_model` was set up):
# ppo_trainer.step([input_ids], [response_ids], [reward])

print("\n--- End of Conceptual Demonstration ---")


### Interpreting the Code and Practical Considerations

The provided code offers a highly simplified, conceptual walkthrough of the RLHF pipeline's two main stages: Reward Modeling and PPO Finetuning. It's crucial to understand what this demonstration illustrates and what it abstracts away.

#### Reward Model Training Interpretation

*   **Synthetic Data**: We created a tiny `preference_data` list. In reality, this dataset would contain tens to hundreds of thousands of human-annotated comparisons, collected through sophisticated data labeling platforms. Each comparison involves a prompt and at least two model responses, with humans indicating which they prefer. This is the most expensive and time-consuming part of RLHF.
*   **`distilbert-base-uncased` as RM Base**: For demonstration, we used a small, general-purpose model. Production-grade RMs are often finetuned versions of the base LLM itself (e.g., a `Llama-3-7B` model finetuned for reward prediction) or similarly powerful architectures. They are trained to output a single scalar score representing the 'goodness' of a response given a prompt.
*   **Simplified Training Step**: The code shows a single forward and backward pass for the RM. A real RM training would involve many epochs over a large dataset, using advanced optimizers, learning rate schedules, and validation sets to prevent overfitting.
*   **Mock `get_reward` Function**: This function is a placeholder. In a real RLHF setup, the `get_reward` function would invoke the *actual trained Reward Model* to predict a score for a given prompt-response pair.

#### PPO Finetuning Interpretation

*   **`gpt2` as Policy Model**: We used `gpt2` as our policy model. In a real RLHF pipeline, this would be the SFT-tuned LLM (e.g., `Llama-3-8B-SFT`).
*   **`trl` Library**: The `trl` (Transformer Reinforcement Learning) library from Hugging Face is a modern, 2026-ready toolkit specifically designed to simplify RLHF implementation. It provides `PPOTrainer` and other utilities that abstract away much of the complexity of PPO.
*   **Conceptual PPO Step**: The code demonstrates the *flow*: a prompt is given, the policy model generates a response, the (mock) Reward Model assigns a reward, and then these components would be fed into the `PPOTrainer.step()` method. A real PPO training loop would involve:
    *   Generating a batch of responses for a batch of prompts.
    *   Calculating rewards for all responses using the RM.
    *   Calculating log probabilities of actions (tokens) from both the current policy and a reference policy (a frozen copy of the policy before the PPO update, used to compute KL divergence for stability).
    *   Computing advantages and value estimates.
    *   Performing multiple mini-batch updates (PPO epochs) on the policy model using the PPO objective, which balances maximizing reward with staying close to the previous policy (controlled by `target_kl`).
*   **Performance Trade-offs**: RLHF is computationally intensive. Training the Reward Model requires significant GPU resources and human annotation effort. PPO finetuning is even more demanding, as it involves running the LLM for generation, the RM for scoring, and then performing policy updates, often for millions of tokens. This process typically requires multiple high-end GPUs (e.g., NVIDIA H100s or equivalent) and can take days or weeks for large models.
*   **Hyperparameter Tuning**: PPO has many hyperparameters (`learning_rate`, `target_kl`, `ppo_epochs`, `mini_batch_size`, etc.) that are crucial for stable and effective training. Tuning these is an art and a science, often requiring extensive experimentation.

#### Typical Use Cases

RLHF is primarily used for:

*   **LLM Alignment**: Making LLMs more helpful, harmless, and honest, reducing biases, and improving adherence to instructions.
*   **Safety and Ethics**: Guiding models away from generating toxic, biased, or unsafe content.
*   **Personalization and Style Transfer**: Aligning models to specific user preferences or stylistic requirements.
*   **Domain Adaptation**: Finetuning models for specific domains where human preferences might differ from general-purpose data.

By understanding this pipeline, ML engineers can effectively contribute to aligning the next generation of AI models with human values and intentions.


### Resources

To dive deeper into RLHF and its practical implementation, explore the following resources:

*   **Hugging Face `trl` Library Documentation**: The official documentation for the `trl` library is an excellent starting point for implementing RLHF, DPO, and other preference-based learning methods. It includes examples and guides for various models.
    *   [Hugging Face `trl` Documentation](https://huggingface.co/docs/trl/en/index)
    *   [PPO Trainer Example](https://huggingface.co/docs/trl/en/ppo_trainer)

*   **Hugging Face `transformers` Library Documentation**: Essential for working with pre-trained LLMs and their tokenizers.
    *   [Hugging Face `transformers` Documentation](https://huggingface.co/docs/transformers/index)

*   **PyTorch Documentation**: The underlying deep learning framework for `transformers` and `trl`.
    *   [PyTorch Official Website](https://pytorch.org/docs/stable/index.html)

*   **Original PPO Paper**: For a deeper theoretical understanding of the Proximal Policy Optimization algorithm.
    *   [Proximal Policy Optimization Algorithms (OpenAI, 2017)](https://arxiv.org/abs/1707.06347)

*   **RLHF Research Papers**: Explore the foundational and recent advancements in RLHF.
    *   [Training a Helpful and Harmless Assistant with Reinforcement Learning from Human Feedback (Anthropic, 2022)](https://arxiv.org/abs/2204.05862)
    *   [Aligning Language Models to Follow Instructions (OpenAI, 2022 - InstructGPT paper)](https://arxiv.org/abs/2203.02155)

*   **Google AI Studio / Gemini API**: While not directly about RLHF implementation, understanding how aligned models are deployed and interact with users provides context for the 'why' behind RLHF.
    *   [Google AI Studio](https://aistudio.google.com/)
    *   [Gemini API Documentation](https://ai.google.dev/docs/gemini_api_overview)

These resources will equip you with both the theoretical knowledge and practical tools to implement and advance RLHF techniques in your own projects.
